# Level セマンティックブロック層

**前提**: UE Editor で `/Game/Maps/Level` を開き **PIE 実行中**。

## 領域

- 角 A: (-11.40, -21.39) m、角 B: (60.13, 58.60) m（カメラ XY）
- 対角矩形 + **各辺 3 m** 外側
- ブロック下面 Z: **6477.1 cm**（角 A、±0.15 m 自動補正・最大 10 回）

## デフォルト

PIE 検証用に **5×5 サブグリッド** `(gx,gy)=(1..5)` のみ配置。全領域は `pie_subgrid=None` + `allow_large_region=True`。

カーネル: `conda activate simworld`

In [ ]:
import importlib
import sys
from pathlib import Path
from typing import Optional

from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_level_dir = _root / "dev" / "grid_env_level_semantic"
_g10k_dir = _root / "dev" / "grid_env_10k"
_geh_dir = _root / "dev" / "grid_env_hri"
for p in (_root, _level_dir, _g10k_dir, _geh_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

ucv: Optional[UnrealCV] = None
print(f"[Paths] {_level_dir}")

In [ ]:
import grid_env_level_semantic as lvl
import level_camera_probe as lcp

importlib.reload(lvl)

if not lcp.wait_for_ue_port(120.0):
    raise RuntimeError("UnrealCV port 9000 — start Level PIE first.")

ucv, _ = lvl.ensure_connection()
print("OK: connected")

In [ ]:
# --- 実行パラメータ ---
PIE_SUBGRID = (1, 1, 5, 5)   # PIE 検証: 5×5。全領域は None
ALLOW_LARGE_REGION = False    # 全領域時 True 必須（~74k cells）
CLEANUP_BEFORE = True

region = lvl.default_level_region()
print(f"grid full size: {region.grid_nx}x{region.grid_ny} = {region.cell_count} cells")
print(f"block_bottom_z = {region.block_bottom_z_cm:.1f} cm")

result = lvl.run_level_semantic_layer(
    ucv,
    region=region,
    cleanup_before=CLEANUP_BEFORE,
    allow_large_region=ALLOW_LARGE_REGION,
    pie_subgrid=PIE_SUBGRID,
)
print(f"done: registry={result.registry_path} height_steps={result.height_adjust_steps}")